In [1]:
!pip install biopython
!pip install numpy
!pip install pandas
!pip install scipy
!pip install openpyxl
import numpy as np
from Bio.Seq import Seq
import pandas as pd
import itertools
import re
from scipy.spatial import distance
import json
from Bio.SeqUtils import MeltingTemp as mt
import random
import pprint
import string
import primer3


In [2]:
#Import primers
orthogonal_F = pd.read_excel('./RR_orthogonalFv1_plate.xlsx')
orthogonal_R = pd.read_excel('./RR_orthogonalRv1_plate.xlsx')


In [3]:
#Isolate 20-nt variable primer binding region and only use that to align to sequences
orthogonal_F['PrimerEnd'] = orthogonal_F.Sequence.apply(lambda x: x[-20:])
orthogonal_R['PrimerEnd'] = orthogonal_R.Sequence.apply(lambda x: x[-20:])


In [4]:
#Check for BsaI sites and remove any primers with them
bsaI_seq = Seq('GGTCTC')
orthogonal_F['BsaI_Site_Present'] = orthogonal_F.Sequence.apply(lambda x: ( (str(bsaI_seq) in x) | (str(bsaI_seq.reverse_complement()) in x) ))
orthogonal_R['BsaI_Site_Present'] = orthogonal_R.Sequence.apply(lambda x: ( (str(bsaI_seq) in x) | (str(bsaI_seq.reverse_complement()) in x) ))


In [5]:
# Adapted from Willow Coyote-Maestas' DIMPLE paper, cite doi: 10.1186/s13059-023-02880-6
def check_nonspecific(primer, fragment, Tm_verb = 20, Tm_rem = 28, primer3_shift = 4, verbose=True):
    non = []
    # Forward
    for i in range(len(fragment) - len(primer)):  # Scan each position
        match = [
            primer[j].lower() == fragment[i + j].lower() for j in range(len(primer))
        ]
        first = 10
        for k in range(len(match) - 3):
            if (match[k] and match[k + 1] and match[k + 3]) or (
                match[k] and match[k + 1] and match[k + 2]
            ):
                first = k
                break
        if (
            sum(match[first:]) > len(primer[first:]) * 0.8
            and sum(match[first:]) > 6
            and match[-1]
        ):  # string compare - sum of matched nt is greater than 80%
            try:
                melt = mt.Tm_NN(
                    primer[first:],
                    c_seq=fragment[i + first : i + len(primer)].complement(),
                    nn_table=mt.DNA_NN2,
                    de_table=mt.DNA_DE1,
                    imm_table=mt.DNA_IMM1,
                )
                if verbose == True:
                    if melt > Tm_verb:
                        print("Found non-specific match at " + str(i + 1) + "bp:")
                        print(" match:" + fragment[i : i + len(primer)])
                        print("primer:" + primer + " Tm:" + str(round(melt, 1)))
                if melt > Tm_rem:
                    non.append(True)
            except ValueError as valerr:
                # use primer3 instead, as mt.DNA_NN2 table does not have enough information to compute Tm
                result = primer3.calcHeterodimer(str(primer[first:]), 
                                                 str(fragment[i + first : i + len(primer)].complement())
                                                )
                melt = result.tm
                if verbose == True:
                    if melt > Tm_verb-primer3_shift:
                        print("Found non-specific match using Primer3 at " + str(i + 1) + "bp:")
                        print(" match:" + fragment[i : i + len(primer)])
                        print("primer:" + primer + " Tm:" + str(round(melt, 1)))
                if melt > Tm_rem-primer3_shift:
                    non.append(True)
    # Reverse
    fragment = fragment.reverse_complement()
    for i in range(len(fragment) - len(primer)):
        match = [
            primer[j].lower() == fragment[i + j].lower() for j in range(len(primer))
        ]
        first = 10
        for k in range(0, len(match) - 3, 1):
            if match[k] and match[k + 1] and match[k + 3]:
                first = k
                break
        if (
            sum(match[first:]) > len(primer[first:]) * 0.8
            and sum(match[first:]) > 6
            and match[-1]
        ):  # string compare - sum of matched nt is greater than 80%
            try:
                melt = mt.Tm_NN(
                    primer[first:],
                    c_seq=fragment[i + first : i + len(primer)].complement(),
                    nn_table=mt.DNA_NN2,
                    de_table=mt.DNA_DE1,
                    imm_table=mt.DNA_IMM1,
                )
                if verbose == True:
                    if melt > Tm_verb:
                        print("Found non-specific match at " + str(i + 1) + "bp:")
                        print(" match:" + fragment[i : i + len(primer)])
                        print("primer:" + primer + " Tm:" + str(melt))
                if melt > Tm_rem:
                    non.append(True)
            except ValueError as valerr:
                # use primer3 instead, as mt.DNA_NN2 table does not have enough information to compute Tm
                result = primer3.calcHeterodimer(str(primer[first:]), 
                                                 str(fragment[i + first : i + len(primer)].complement())
                                                )
                melt = result.tm
                if verbose == True:
                    if melt > Tm_verb-primer3_shift:
                        print("Found non-specific match using Primer3 at " + str(i + 1) + "bp:")
                        print(" match:" + fragment[i : i + len(primer)])
                        print("primer:" + primer + " Tm:" + str(round(melt, 1)))
                if melt > Tm_rem-primer3_shift:
                    non.append(True)
    return sum(non)


In [6]:
# Define genes

# BRAF
braf_part1 = Seq('ATGgctgcgctgagcGGTGGaGGTGGaGGCGGTGCAGAACCCGGGCAAGCATTGTTCAATGGGGATATGGAACCTGAAGCTGGCGCAGGAGCCGGAgccgcggcctcttcggctGCCGATCCTGCTATTCCTGAGGAGGTGTGGAACATCAAACAGATGATTAAACTGACGCAAGAGCATATTGAAGCCCTGCTTGATAAGTTCGGGGGCGAGCATAACCCCCCCAGCATTTATCTGGAGGCATATGAGGAATATACGTCTAAACTCGACGCCCTTCAACAGCGGGAACAGCAGCTCCTTGAAAGCCTGGGCAATGGGACGGATTTCTCAGTTAGTTCCAGTGCCAGTATGGATACAGTGACCTCATCATCCAGTTCCTCCCTGTCTGTCCTGCCTTCTTCTCTCTCTGTTTTCCAAAACCCGACTGATGTAGCAAGGTCTAATCCTAAAAGCCCACAAAAGCCGATTGTTCGGGTATTCCTGCCAAACAAACAAAGAACAGTTGTCCCAGCGCGATGCGGCGTCACTGTTCGAGATAGTCTCAAAAAAGCACTGATGATGCGGGGATTGATTCCCGAATGCTGCGCAGTTTACCGGATTCAGGATGGGGAGAAAAAACCAATAGGTTGGGACACGGATATAAGTTGGCTTACAGGGGAGGAGCTTCACGTTGAGGTTCTGGAGAATGTTCCCTTGACAACACATAATTTTGTCAGAAAGACGTTCTTCACTTTGGCTTTCTGTGACTTTTGTAGAAAGCTGCTCTTtCAGGGCTTTCGATGTCAGACCTGCGGGTATAAATTCCACCAGCGGTGCTCCACAGAGGTACCTTTGATGTGTGTCAACTATGATCAACTTGACCTTCTTTTCGTCAGTAAGTTCTTTGAACACCATCCGATCCCTCAGGAGGAAGCCTCCCTTGCCGAGACTGCCTTGACCAGTGGCTCCTCTCCTTCTGCACCTGCATCCGACAGTATAGGCCCGCAAATATTGACCTCACCGAGCCCCAGTAAATCTATTCCGATCCCGCAGCCCTTTCGCCCTGCGGACGAAGATCATAGGAATCAGTTTGGACAGCGCGACAGATCCTCCAGTGCGCCGAACGTACACATAAATACAATTGAACCTGTTAATATAGACGATTTGATTCGCGACCAAGGGTTTAGGGGTGACGGAGGGTCAACCACCGGTTTGTCTGCTACTCCACCAGCTTCTCTGCCAGGcTCTCTCACAAATGTTAAAGCATTGCAAAAATCCCCTGGACCTCAACGAGAAAGG'.upper())
braf_part2 = Seq('AGGAAaAGCAGCTCATCTAGCGAGGACCGCAACAGAATGAAGACTCTCGGTAGAAGAgactcgagtGACGACTGGGAAATACCAGACGGTCAAATCACGGTCGGTCAGCGGATCGGATCAGGCTCCTTCGGGACTGTATATAAAGGTAAATGGCACGGCGACGTTGCGGTCAAAATGCTGAACGTTACAGCACCAACCCCCCAACAGCTTCAAGCGTTCAAGAACGAAGTCGGGGTACTGCGCAAAACTCGGCATGTCAATATATTGCTGTTCATGGGTTACTCAACCAAGCCTCAACTCGCCATAGTTACCCAGTGGTGTGAAGGCAGCTCACTTTACCATCACCTGCACATAATAGAaACCAAGTTCGAGATGATCAAACTCATTGACATTGCGCGGCAAACAGCGCAGGGGATGGACTATCTGCATGCAAAGTCTATTATCCATAGAGAtCTCAAGTCAAACAACATATTCTTGCACGAGGATCTCACCGTTAAAATTGGAGACTTTGGcCTCGCAACTGTTAAGTCACGGTGGAGTGGGTCACATCAGTTCGAGCAGCTGTCCGGCAGCATACTGTGGATGGCTCCAGAGGTCATTCGCATGCAGGATAAGAACCCTTATTCTTTCCAATCCGATGTTTATGCATTTGGCATCGTCCTCTATGAGCTGATGACCGGACAACTCCCTTACAGCAACATCAACAATCGAGAtCAGATCATCTTCATGGTCGGGCGAGGATACCTCAGCCCCGATCTCTCAAAGGTTCGATCAAATTGCCCTAAAGCGATGAAACGGCTTATGGCGGAGTGTTTGAAAAAGAAACGCGACGAACGCCCTTTGTTCCCTCAAATCTTGGCATCAATCGAGTTGCTGGCTAGAAGTCTTCCAAAAATACACAGAAGCGCATCCGAGCCAAGCCTCAATCGGGCAGGATTCCAGACCGAGGATTTCTCTCTTTATGCCTGCGCATCTCCAAAGACACCCATACAGGCCGGCGGGTACggCgcCtttcctgtgcacTAG'.upper())

# KRAS
kras = Seq('atgactgaatataaacttgtggtagttggagctgggggcgtaggcaagagtgccttgacgatacagctaattcagaatcattttgtggacgaatatgatccaacaatagaggattcctacaggaagcaagtagtaattgatggagaaacctgtctcttggatattctcgacacagcaggtcaagaggagtacagtgcaatgagggaccagtacatgaggactggggagggctttctttgtgtatttgccataaataatactaaatcatttgaagatattcaccattatagagaacaaattaaaagagttaaggactctgaagatgtacctatggtcctagtaggaaataaatgtgatttgccttccagaacagtagacacaaaacaggctcaggacttagcaagaagttatggaattccttttattgaaacatcagcaaagacaagacagggtgttgatgatgccttctatacattagttcgagaaattcgaaaacataaagaaaagatgagcaaagatggtaaaaagaagaaaaagaagtcaaagacaaagtgtgtaattatgTAG'.upper())

# MRAS
mras = Seq('ATGGCCACATCAGCCGTCCCATCCGACAACCTTCCCACATACAAATTGGTCGTGGTTGGGGATGGAGGGGTGGGTAAGTCAGCCCTTACTATCCAATTCTTCCAAAAAATCTTCGTGCCCGACTACGATCCGACCATTGAAGACAGTTACCTGAAGCATACAGAGATAGACAACCAATGGGCAATTTTGGATGTGCTGGATACAGCGGGACAAGAGGAGTTCTCCGCCATGCGCGAACAATATATGAGGACAGGGGACGGTTTTCTTATAGTGTATTCCGTGACTGATAAAGCGAGTTTCGAGCACGTGGATCGCTTTCATCAGCTGATACTTCGCGTCAAGGACAGAGAGAGCTTCCCGATGATACTCGTGGCTAATAAAGTGGATTTGATGCACCTTCGAAAGATTACACGCGAACAAGGCAAAGAAATGGCGACAAAACATAATATTCCTTATATAGAAACGTCTGCGAAGGACCCGCCGCTTAACGTCGATAAAGCGTTCCACGATTTGGTTAGGGTCATACGCCAGCAGATCCCGGAGAAATCCCAGAAGAAGAAGAAAAAGACTAAATGGCGGGGTGATAGGGCCACAGGGACACACAAGCTTCAGTGTGTAATTCTCTAG'.upper())

# EGFR split
egfr_cterm_part1 = Seq('AGGCGCCACATCGTTCGGAAGCGCACGCTGCGGAGGCTGCTGCAGGAGAGGGAGCTTGTGGAGCCTCTTACACCCAGTGGAGAAGCTCCCAACCAAGCTCTCTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGTGCTGGGCTCCGGTGCGTTCGGCACGGTGTATAAGGGACTCTGGATCCCAGAAGGTGAGAAAGTTAAAATTCCCGTCGCTATCAAGGAATTAAGAGAAGCAACATCTCCGAAAGCCAACAAGGAAATCCTCGATGAAGCCTACGTGATGGCCAGCGTGGACAACCCCCACGTGTGCCGCCTGCTGGGCATCTGCCTCACCTCCACCGTGCAACTCATCACGCAGCTCATGCCCTTCGGCTGCCTCCTGGACTATGTCCGGGAACACAAAGACAATATTGGCTCCCAGTACCTGCTCAACTGGTGTGTGCAGATCGCAAAGGGCATGAACTACTTGGAGGACCGTCGCTTGGTGCACCGCGACCTGGCAGCCAGGAACGTACTGGTGAAAACACCGCAGCATGTCAAGATCACAGATTTTGGGCTGGCCAAACTGCTGGGTGCGGAAGAGAAAGAATACCATGCAGAAGGAGGCAAAGTGCCTATCAAGTGGATGGCATTGGAATCAATTTTACACAGAATCTATACCCACCAGAGTGATGTCTGGAGCTACGGGGTGACCGTTTGGGAGTTGATGACCTTTGGATCCAAGCCATATGACGGAATCCCTGCCAGCGAGATCTCCTCCATCCTGGAGAAAGGAGAACGCCTCCCTCAGCCACCCATATGTACCATCGATGTCTACATGATCATGGTCAAGTGCTGGATGATAGACGCAGATAGTCGCCCAAAGTTCCGTGAGTTGATCATCGAATTCTCCAAAATGGCCCGAGAtCCCCAGCGCTACCTTGTCATTCAGGGGGATGAAAGAATGCATTTGCCAAGTCCTACAGACTCCAACTTCTACCGTGCCCTGATGGATGAAGAAGACATGGACGACGTGGTGGATGCCGACGAGTACCTCATCCCACAGCAGGGCTTCTTCAGCAGCCCCTCCACGTCACGGACTCCCCTC'.upper())
egfr_cterm_part2 = Seq('CTCCTGAGCTCTCTGAGTGCAACCAGCAACAATTCCACCGTGGCTTGCATTGATAGAAATGGGCTGCAAAGCTGTCCCATCAAGGAAGACAGCTTCTTGCAGCGATACAGCTCAGACCCCACAGGCGCCTTGACTGAGGACAGCATAGACGACACCTTCCTCCCAGTGCCTGAATACATAAACCAGTCCGTTCCCAAAAGGCCCGCTGGCTCTGTGCAGAATCCTGTCTATCACAATCAGCCTCTGAACCCCGCGCCCAGCAGAGAtCCACACTACCAGGACCCCCACAGCACTGCAGTGGGCAACCCCGAGTATCTCAACACTGTCCAGCCCACCTGTGTCAACAGCACATTCGACAGCCCTGCCCACTGGGCCCAGAAAGGCAGCCACCAAATTAGCCTGGACAACCCTGACTACCAGCAGGACTTCTTTCCCAAGGAAGCCAAGCCAAATGGCATCTTTAAGGGCTCCACAGCTGAAAATGCAGAATACCTAAGGGTCGCGCCACAAAGCAGTGAATTTATTGGAGCAacg'.upper())

# ERBB2 split
erbb2_cterm_part1 = Seq('tacacgatgcggagactgctgcaggaaacggagctggtggagccgctgacacctagcggagcgatgcccaaccaggcgcagatgcggatcctgaaagaAacggagctgaggaaggtgaaggtgcttggatctggcgcttttggcacagtctacaagggcatctggatccctgatggggagaatgtgaaaattccagtggccatcaaagtgttgagggaaaacacatcccccaaagccaacaaagaaatcttagacgaagcatacgtgatggctggtgtgggctccccatatgtctcccgccttctgggcatctgcctgacatccacggtgcagctggtgacacagcttatgccctatggctgcctcttagaccatgtccgggaaaaccgcggacgcctgggctcccaggacctgctgaactggtgtatgcagattgccaaggggatgagctacctggaggatgtgcggctcgtacacagggacttggccgctcggaacgtgctggtcaagagtcccaaccatgtcAAAATAACAGATTTCGGTCTGGCGCGACTTCTGGATATAGATGAAACGGAGTATCATGCCGATGGTGGCAAAGTGCCTATCAAATGGATGGCATTGGAAAGCATTCTGCGACGACGATTCACGCACCAGTCTGATGTGTGGAGTTACGGCGTTACGGTATGGGAGCTTATGACTTTTGGTGCGaaaccttacgatgggatcccagcccgggagatccctgacctgctggaaaagggggagcggctgccccagccccccatctgcaccattgatgtctacatgatcatggtcaaatgttggatgattgactctgaatgtcggccaagattccgggagttggtgtctgaattctcccgcatggccagggacccccagcgctttgtggtcatccagaatgaggacttgggcccagccagtcccttggacagcaccttctaccgctcactgctggaggacgatgacatgggggacctggtggatgctgaggagtatctggtaccccagcagggcttcttctgtcca'.upper())
erbb2_cterm_part2 = Seq('ccagaccctgccccgggcgctgggggcatggtccaccacaggcaccgcagctcatctaccaggagtggcggtggggacctgacactagggctggagccctctgaagaggaggcccccagAtctccactggcaccctccgaaggggctggctccgatgtatttgatggtgacctgggaatgggggcagccaaggggctgcaaagcctccccacacatgaccccagccctctacagcggtacagtgaggaccccacagtacccctgccctctgagactgatggctacgttgcccccctgacctgcagcccccagcctgaatatgtgaaccagccagatgttcggccccagcccccttcgccccgagagggccctctgcctgctgcccgacctgctggtgccactctggaaaggcccaagactctctccccagggaagaatggggtcgtcaaagacgtttttgcctttgggggtgccgtggagaaccccgagtacttgacaccccagggaggagctgcccctcagccccaccctcctcctgccttcagcccagccttcgacaacctctattactgggaccaggacccaccagagcggggggctccacccagcaccttcaaagggacacctacggcagagaacccagagtacctgggtctggacgtgccagtgacg'.upper())

# SHP2
shp2_part1 = Seq('atgacatcgcggagatggtttcacccaaatatcactggtgtggaggcagaaaacctactgttgacaagaggagttgatggcagttttttggcaaggcctagtaaaagtaaccctggagacttcacactttccgttagaagaaatggagctgtcacccacatcaagattcagaacactggtgattactatgacctgtatggaggggagaaatttgccactttggctgagttggtccagtattacatggaacatcacgggcaattaaaagagaagaatggagatgtcattgagcttaaatatcctctgaactgtgcagatcctacctctgaaaggtggtttcatggacatctctctgggaaagaagcagagaaattattaactgaaaaaggaaaacatggtagttttcttgtacgagagagccagagccaccctggagattttgttctttctgtgcgcactggtgatgacaaaggggagagcaatgacggcaagtctaaagtgacccatgttatgattcgctgtcaggaactgaaatacgacgttggtggaggagaacggtttgattctttgacagatcttgtggaacattataagaagaatcctatggtggaaacattgggtacagtactacaactcaagcagccccttaacacgactcgtataaatgct'.upper())
shp2_part2 = Seq('gctgctgaaatagaaagcagagttcgagaactaagcaaattagctgaAacAacagataaagtcaaacaaggcttttgggaagaatttgagacactacaacaacaggagtgcaaacttctctacagccgaaaagagggtcaaaggcaagaaaacaaaaacaaaaatagatataaaaacatcctgccctttgatcataccagggttgtcctacacgatggtgatcccaatgagcctgtttcagattacatcaatgcaaatatcatcatgcctgaatttgaaaccaagtgcaacaattcaaagcccaaaaagagttacattgccacacaaggctgcctgcaaaacacggtgaatgacttttggcggatggtgttccaagaaaactcccgagtgattgtcatgacaacgaaagaagtggagagaggaaagagtaaatgtgtcaaatactggcctgatgagtatgctctaaaagaatatggcgtcatgcgtgttaggaacgtcaaagaaagcgccgctcatgactatacgctaagagaacttaaactttcaaaggttggacaagggaatacggagagaacggtctggcaataccactttcggacctggccggaccacggcgtgcccagcgaccctgggggcgtgctggacttcctggaggaggtgcaccataagcaggagagcatcatggatgcagggccggtcgtggtgcactgcagtgctggaattggccggacagggacgttcattgtgattgatattcttattgacatcatcagagagaaaggtgttgactgcgatattgacgttcccaaaaccatccagatggtgAGAAGCcagaggtcagggatggtccagacagaagcacagtaccgatttatctatatggcggtccagcattatattgaaacactacagcgcaggattgaagaGgagcagaaaagcaagaggaaagggcacgaatatacaaatattaagtattctctagcggaccagacgagtggagatcagagccctctcccgccttgtactccaacgccaccctgtgcagaaatgagagaagacagtgctagagtctatgaaaacgtgggcctgatgcaacagcagaaaagtttcagaTAG'.upper())

# SOS2
sos2_cterm = Seq('aacttgcaaagtagaagtggcatccccattattaaaggaggaactgtagtgaaattaattgaaaggttaacatatcatatgtatgcagatcccaattttgttcgtacttttcttaccacatatcgttcattttgtaaaccacaggaattgctgagcttactgattgaacggtttgaaattccagagccagaacctactgacgcagacaaattggcaatagagaaaggcgagcagccaatcagtgcagaccttaaaagatttcgcaaggaatatgtccaaccagtacaacttaggatcttaaatgtatttcggcattgggttgaacatcatttttatgactttgaaagagacttggaattgcttgaaagactagaatccttcatttcaagtgtaagagggaaagctatgaaaaaatgggtagagtcaattgctaagatcatcaggaggaagaagcaagctcaggcaaatggagtaagccataatattacctttgaaagtccacctccaccaattgaatggcatatcagcaaaccaggacagtttgaaacatttgatctcatgacacttcatccaatagaaattgcacgtcagctgacacttttggagtctgatctttacaggaaagttcaaccgtctgaacttgtagggagtgtgtggaccaaagaagataaagaaataaattctccaaatttattaaaaatgattcgccataccacaaatctcaccctctggtttgaaaaatgcattgtggaagcagaaaattttgaagaacgggtggcagtactaagtagaattatagaaattctgcaagtttttcaagatttgaataatttcaatggcgtattggagatagtcagtgcagtaaattcagtgtcagtatacagactagaccatacctttgaggcactgcaggaaaggaaaaggaaaattttggacgaagctgtggaattaagtcaagatcactttaaaaaatacctagtaaaacttaagtcaatcaatccaccttgtgtgcctttttttggaatatatttaacaaatattctgaagaccgaagaagggaataatgattttttaaaaaagaaagggaaagatttaatcaatttcagtaagaggaggaaagtagctgaaattactggagaaattcagcagtatcagaatcagccttactgtttacggatagaaccagatatgaggagattctttgaaaaccttaaccccatgggaagtgcatctgaaaaagagtttacagattatttgttcaacaagtcactagaaattgaacctcgaaactgcaaacagccacctcgatttcctaggaaatcaactttttcc'.upper())

# ARAF
araf_part1 = Seq('atggagccaccacggggcccccctgccaatggggccgagccatcccgggcagtgggcaccgtcaaagtatacctgcccaacaagcaacgcacggtggtgactgtccgggatggcatgagtgtctacgactctctagacaaggccctgaaggtgcggggtctaaatcaggactgctgtgtggtctaccgactcatcaagggacgaaagacggtcactgcctgggacacagccattgctcccctggatggcgaggagctcattgtcgaggtccttgaagatgtcccgctgaccatgcacaattttgtacggaagaccttcttcagcctggcgttctgtgacttctgccttaagtttctgttccatggcttccgttgccaaacctgtggctacaagttccaccagcattgttcctccaaggtccccacagtctgtgttgacatgagtaccaaccgccaacagttctaccacagtgtccaggatttgtccggaggctccagacagcatgaggctccctcgaaccgccccctgaatgagttgctaaccccccagggtcccagcccccgcacccagcactgtgacccggagcacttccccttccctgccccagccaatgcccccctacagcgcatccgctccacgtccactcccaacgtccatatggtcagcaccacggcccccatggactccaacctcatccagctcactggccagagtttcagcactgatgctgccggtagtagaggaggtagtgatggaaccccccgggggagccccagcccagccagcgtgtcctcggggaggaagtccccacattccaagtcaccagcagagcagcgcgagcggaagtccttggccgatgacaagaagaaagtgaagaacctggggtaccgg'.upper())
araf_part2 = Seq('cgggactcaggctattactgggaggtaccacccagtgaggtgcagctgctgaagaggatcgggacgggctcgtttggcaccgtgtttcgagggcggtggcatggcgatgtggccgtgaaggtgctcaaggtgtcccagcccacagctgagcaggcccaggctttcaagaatgagatgcaggtgctcaggaagacgcgacatgtcaacatcttgctgtttatgggcttcatgacccggccgggatttgccatcatcacacagtggtgtgagggctccagcctctaccatcacctgcatgtggccgacacacgcttcgacatggtccagctcatcgacgtggcccggcagactgcccagggcatggactacctccatgccaagaacatcatccaccgagatctcaagtctaacaacatcttcctacatgaggggctcacggtgaagatcggtgactttggcttggccacagtgaagactcgatggagcggggcccagcccttggagcagccctcaggatctgtgctgtggatggcagctgaggtgatccgtatgcaggacccgaacccctacagcttccagtcagacgtctatgcctacggggttgtgctctacgagcttatgactggctcactgccttacagccacattggctgccgtgaccagattatctttatggtgggccgtggctatctgtccccggacctcagcaaaatctccagcaactgccccaaggccatgcggcgcctgctgtctgactgcctcaagttccagcgggaggagcggcccctcttcccccagatcctggccacaattgagctgctgcaacggtcactccccaagattgagcggagtgcctcggaaccctccttgcaccgcacccaggccgatgagttgcctgcctgcctactcagcgcagcccgccttgtgcctTAG'.upper())

# CRAF
craf_part1 = Seq('atggagcacatacagggagcttggaagacgatcagcaatggttttggattcaaagatgccgtgtttgatggctccagctgcatctctcctacaatagttcagcagtttggctatcagcgccgggcatcagatgatggcaaactcacagatccttctaagacaagcaacactatccgtgttttcttgccgaacaagcaaagaacagtggtcaatgtgcgaaatggaatgagcttgcatgactgccttatgaaagcactcaaggtgaggggcctgcaaccagagtgctgtgcagtgttcagacttctccacgaacacaaaggtaaaaaagcacgcttagattggaatactgatgctgcgtctttgattggagaagaacttcaagtagatttcctggatcatgttcccctcacaacacacaactttgctcggaagacgttcctgaagcttgccttctgtgacatctgtcagaaattcctgctcaatggatttcgatgtcagacttgtggctacaaatttcatgagcactgtagcaccaaagtacctactatgtgtgtggactggagtaacatcagacaactcttattgtttccaaattccactattggtgatagtggagtcccagcactaccttctttgactatgcgtcgtatgcgagagtctgtttccaggatgcctgttagttctcagcacagatattctacacctcacgccttcacctttaacacctccagtccctcatctgaaggttccctctcccagaggcagaggtcgacatccacacctaatgtccacatggtcagcaccaccctgcctgtggacagcaggatgattgaggatgcaattcgaagtcacagcgaatcagcctcaccttcagccctgtccagtagccccaacaatctgagcccaacaggctggtcacagccgaaaacccccgtgccagcacaaagagagcgggcaccagtatctgggacccaggagaaaaacaaaattaggcctcgtggacagaga'.upper())
craf_part2 = Seq('agagattcaagctattattgggaaatagaagccagtgaagtgatgctgtccactcggattgggtcaggctcttttggaactgtttataagggtaaatggcacggagatgttgcagtaaagatcctaaaggttgtcgacccaaccccagagcaattccaggccttcaggaatgaggtggctgttctgcgcaaaacacggcatgtgaacattctgcttttcatggggtacatgacaaaggacaacctggcaattgtgacccagtggtgcgagggcagcagcctctacaaacacctgcatgtccaggaAaccaagtttcagatgttccagctaattgacattgcccggcagacggctcagggaatggactatttgcatgcaaagaacatcatccatagagacatgaaatccaacaatatatttctccatgaaggcttaacagtgaaaattggagattttggtttggcaacagtaaagtcacgctggagtggttctcagcaggttgaacaacctactggctctgtcctctggatggccccagaggtgatccgaatgcaggataacaacccattcagtttccagtcggatgtctactcctatggcatcgtattgtatgaactgatgacgggggagcttccttattctcacatcaacaaccgagatcagatcatcttcatggtgggccgaggatatgcctccccagatcttagtaagctatataagaactgccccaaagcaatgaagaggctggtagctgactgtgtgaagaaagtaaaggaagagaggcctctttttccccagatcctgtcttccattgagctgctccaacactctctaccgaagatcaaccggagcgcttccgagccatccttgcatcgggcagcccacactgaggatatcaatgcttgcacgctgaccacgtccccgaggctgcctgtcttcTAG'.upper())

# KSR1
ksr1_part1 = Seq('ATGGACAGAGCCGCATTGAGGGCGGCTGCCATGGGAGAGAAGAAAGAAGGGGGTGGTGGTGGAGATGCTGCGGCAGCTGAGGGTGGCGCAGGAGCAGCGGCAAGCCGCGCATTGCAACAGTGCGGGCAACTTCAGAAATTGATCGATATCAGTATCGGAAGCCTGAGAGGTCTGCGGACCAAGTGCGCGGTATCAAACGACCTCACCCAGCAAGAAATCAGGACGCTTGAGGCAAAACTTGTACGATATATATGTAAGCAGCGCCAATGTAAATTGAGCGTTGCCCCAGGTGAAAGGACCCCTGAGCTGAATTCCTACCCCCGATTCTCCGACTGGCTCTATACATTTAACGTAAGGCCTGAAGTTGTACAGGAGATCCCAAGGGACCTGACCCTCGACGCTCTCCTTGAAATGAATGAAGCCAAAGTCAAGGAAACTCTGCGGAGGTGCGGCGCGTCTGGTGATGAATGCGGTAGACTGCAGTATGCACTGACCTGTCTTAGAAAGGTTACTGGCCTCGGGGGAGAGCACAAAGAGGATAGCAGCTGGAGTAGCCTGGACGCTCGACGCGAAAGCGGATCTGGACCTAGCACAGACACATTGAGCGCTGCGAGTCTGCCTTGGCCTCCTGGGTCATCTCAGCTTGGCCGCGCAGGCAACTCTGCTCAGGGACCGCGGTCAATCTCCGTAAGCGCGCTCCCGGCCAGTGATAGCCCGACCCCATCCTTCTCTGAGGGTTTGAGCGACACCTGTATTCCTCTGCATGCCAGTGGTCGACTCACACCTCGGGCACTGCATTCCTTTATCACCCCTCCAACTACGCCTCAGCTCCGCAGGCACACGAAATTGAAGCCTCCGCGGACACCGCCCCCGCCGTCCCGGAAAGTGTTTCAGCTCCTCCCAAGTTTCCCTACTCTTACTAGGTCTAAATCACACGAATCACAATTGGGTAACAGGATTGACGATGTTAGCAGCATGAGGTTCGACCTCTCACACGGCTCACCACAAATGGTGCGCAGGGACATCGGcCTCTCCGTTACCCATAGATTTTCCACAAAGAGCTGGTTGTCCCAGGTCTGCCATGTTTGCCAGAAGTCAATGATATTCGGGGTGAAATGTAAACACTGCCGACTCAAATGCCACAATAAGTGCACGAAGGAAGCCCCAGCGTGTCGGATATCTTTTCTCCCGTTGACACGCCTCAGGAGGACAGAATCTGTTCCTTCAGACATAAATAACCCGGTTGATCGGGCGGCTGAACCGCATTTCGGAACCTTGCCGAAGGCTCTTACCAAGAAAGAACACCCCCCTGCGATGAACCATTTGGACTCTAGCTCAAACCCCAGTAGTACGACCTCCTCTACGCCTAGCAGCCCTGCCCCATTTCCGACCAGTTCCAACCCATCATCTGCTACTACACCGCCTAACCCAAGTCCC'.upper())
ksr1_part2 = Seq('CCCGGACAACGAGACTCTCGCTTCAACTTCCCGGCAGCCTACTTCATCCATCATCGCCAGCAGTTTATTTTCCCCGTCCCTTCCGCAGGACACTGTTGGAAGTGTCTTTTGATAGCTGAGTCCTTGAAGGAAAATGCCTTTAATATCAGCGCCTTCGCACACGCTGCTCCGCTGCCAGAAGCCGCAGATGGTACCAGATTGGATGATCAACCGAAAGCAGACGTGTTGGAAGCGCATGAGGCGGAGGCGGAGGAGCCGGAGGCAGGcAAGAGCGAGGCCGAAGACGATGAGGACGAGGTTGATGATCTCCCGTCTTCCCGACGGCCCTGGCGAGGTCCTATTAGTCGGAAAGCTTCACAGACGAGTGTCTACCTCCAGGAATGGGATATTCCATTCGAGCAGGTTGAGCTGGGCGAACCAATCGGTCAAGGTCGGTGGGGCCGGGTTCATAGAGGTCGCTGGCATGGGGAGGTGGCTATCCGCCTTTTGGAAATGGATGGACACAACCAAGACCACCTCAAACTGTTCAAGAAAGAAGTGATGAACTACCGACAGACGCGGCACGAGAATGTGGTGCTTTTTATGGGCGCCTGTATGAATCCGCCGCACCTTGCTATAATTACGAGTTTTTGCAAAGGAAGAACGCTTCACAGTTTCGTGAGGGACCCGAAGACATCCTTGGATATAAACAAGACACGACAAATTGCTCAAGAGATTATTAAAGGGATGGGGTATTTGCATGCAAAAGGCATAGTGCATAAAGATCTTAAGTCCAAGAACGTGTTTTATGATAATGGTAAAGTCGTTATTACCGACTTCGGTTTGTTTGGTATCTCTGGTGTCGTACGCGAAGGACGACGAGAGAACCAACTCAAGCTTTCCCACGATTGGCTCTGCTATCTTGCTCCCGAGATAGTTCGCGAAATGACGCCGGGTAAAGATGAAGACCAACTCCCATTTAGCAAGGCTGCCGACGTGTACGCCTTTGGTACCGTCTGGTATGAACTTCAAGCTCGCGATTGGCCCCTCAAGAATCAGGCTGCTGAGGCCAGTATCTGGCAGATAGGAAGCGGTGAAGGTATGAAAAGAGTACTTACATCAGTGTCACTTGGAAAGGAAGTATCTGAAATCCTCTCTGCGTGTTGGGCATTCGATCTTCAAGAAAGACCAAGTTTTTCTCTTCTGATGGACATGCTCGAGAAACTGCCAAAACTTAACCGCCGCCTTTCACACCCAGGACACTTCTGGAAATCAGCCGATATTAATTCCAGCAAAGTCGTACCACGCTTTGAACGGTTCGGCCTGGGTGTCCTGGAATCTAGCAACCCGAAGATGTAG'.upper())

# KSR2
ksr2_part1 = Seq('ATGGACGAGGAGAATATGACCAAAAGTGAGGAGCAACAACCTTTGAGCCTCCAGAAAGCACTTCAGCAGTGTGAACTCGTTCAGAATATGATTGATCTTAGCATATCAAACTTGGAGGGGCTTAGGACGAAATGCGCGACGAGCAACGATCTTACCCAAAAGGAGATAAGAACACTCGAGTCCAAACTTGTGAAATACTTTTCACGCCAACTTAGTTGTAAAAAAAAAGTCGCATTGCAAGAACGGAACGCGGAACTCGATGGGTTTCCGCAATTGAGACACTGGTTTAGGATAGTAGATGTAAGAAAGGAGGTACTGGAGGAAATTTCACCTGGGCAACTTTCTTTGGAAGACCTCTTGGAGATGACCGATGAGCAGGTATGTGAAACGGTCGAAAAATATGGTGCAAACAGAGAAGAATGTGCCAGACTTAATGCCTCACTGTCATGTCTCAGAAACGTGCACATGAGTGGGGGTAATCTCTCTAAGCAAGACTGGACTATACAGTGGCCAACTACCGAAACTGGCAAGGAGAACAACCCAGTTTGCCCGCCTGAGCCTACTCCATGGATTAGGACACACCTCTCTCAATCCCCGAGAGTCCCGTCCAAATGTGTACAACATTACTGCCACACGTCACCTACGCCGGGAGCTCCTGTATACACCCACGTAGACCGGCTGACGGTGGACGCCTATCCTGGCCTTTGTCCTCCTCCCCCACTCGAGAGCGGGCATCGCTCACTTCCTCCAAGCCCACGACAACGACACGCTGTCAGGACACCACCCCGCACACCGAACATTGTTACAACCGTCACGCCTCCGGGAACGCCGCCGATGCGCAAAAAGAACAAACTTAAGCCGCCTGGCACTCCTCCGCCCTCTAGTAGGAAACTTATTCACCTGATCCCTGGCTTCACAGCTCTGCACAGATCCAAATCTCATGAATTCCAACTGGGACACCGGGTGGACGAAGCGCATACGCCAAAAGCCAAGAAGAAGTCTAAACCGCTTAACCTCAAAATACATAGTAGCGTCGGGTCCTGCGAAAATATTCCTTCTCAGCAACGATCTCCACTCTTGTCTGAAAGATCCCTTAGGTCCTTCTTCGTAGGACATGCGCCATTTTTGCCCTCCACACCCCCCGTCCATACAGAAGCAAATTTCTCAGCAAACACACTTTCAGTCCCCCGCTGGAGCCCCCAGATCCCTAGAAGAGATTTGGGTAATAGCATAAAGCACAGGTTTAGCACCAAATATTGGATGTCCCAGACATGTACCGTATGCGGAAAGGGCATGCTGTTCGGTCTTAAATGTAAGAATTGCAAATTGAAGTGTCACAACAAATGCACCAAGGAAGCGCCGCCATGCCATCTGCTCATCATACACCGGGGcGACCCCGCCAGACTCGTTAGAACAGAAAGTGTGCCGTGTGACATTAATAACCCTCTGAGGAAGCCC'.upper())
ksr2_part2 = Seq('CCCCCAAGGTATAGTGACCTTCACATCTCACAAACCCTCCCTAAAACGAATAAGATTAATAAAGACCACATACCAGTCCCGTATCAACCCGATAGCTCCAGCAATCCAAGTTCTACTACTTCCAGCACTCCCTCCTCACCTGCACCGCCACTGCCCCCGTCCGCTACGCCACCTAGTCCTCTGCACCCTAGTCCCCAATGTACCCGGCAGCAGAAGAATTTTAATTTGCCAGCGTCCCATTACTACAAATACAAACAGCAGTTCATATTTCCCGACGTTGTGCCTGTGCCGGAAACGCCAACGAGAGCCCCGCAAGTAATTCTTCATCCAGTCACTAGCAACCCCATCTTGGAAGGTAATCCTCTTCTCCAAATAGAAGTTGAGCCTACTTCAGAAAATGAAGAGGTGCACGATGAAGCCGAAGAGTCTGAGGATGACTTCGAAGAAATGAACCTTTCACTGCTGAGTGCACGGAGTTTTCCTAGAAAAGCCAGTCAGACTTCCATATTCCTGCAAGAGTGGGATATCCCTTTCGAGCAACTCGAAATTGGCGAGCTTATCGGGAAAGGCAGGTTTGGTCAAGTTTATCATGGAAGGTGGCATGGCGAAGTAGCTATTCGCCTTATTGATATTGAACGGGACAACGAGGATCAATTGAAGGCCTTTAAGCGCGAAGTCATGGCTTACCGACAAACCCGACATGAAAACGTCGTTTTGTTTATGGGAGCCTGTATGAGTCCCCCACACCTCGCTATCATAACCAGCCTTTGTAAAGGCCGAACACTCTATTCAGTCGTCCGGGATGCTAAGATCGTACTTGATGTGAATAAAACTCGACAAATTGCTCAAGAAATTGTCAAGGGTATGGGCTATCTCCACGCGAAGGGTATATTGCATAAGGACCTCAAGTCTAAGAACGTGTTTTACGACAACGGTAAAGTCGTCATCACAGATTTCGGTCTGTTTAGCATCAGCGGGGTTCTGCAGGCTGGCAGAAGAGAAGATAAGCTTCGGATTCAAAACGGGTGGCTCTGTCATCTCGCCCCTGAGATTATCAGGCAACTGTCTCCCGACACCGAAGAAGATAAGCTTCCCTTTAGTAAGCATTCCGACGTATTCGCTTTGGGGACGATTTGGTACGAGCTCCATGCTCGAGAGTGGCCTTTTAAGACGCAACCTGCCGAAGCTATCATCTGGCAAATGGGCACAGGCATGAAACCCAATCTGTCTCAAATTGGTATGGGCAAAGAGATCAGTGACATCCTCTTGTTTTGCTGGGCATTTGAGCAGGAGGAAAGGCCTACCTTCACGAAACTCATGGACATGCTCGAGAAGCTCCCCAAGAGAAACAGGAGACTGTCCCATCCAGGACATTTTTGGAAAAGCGCTGAACTGTAG'.upper())

# MEK1
mek1 = Seq('ATGCCCAAAAAGAAACCCACTCCGATTCAGCTGAACCCTGCGCCCGACGGCAGCGCAGTAAACGGGACATCCTCTGCGGAGACTAACCTCGAAGCTTTGCAGAAAAAACTGGAGGAGCTGGAGCTGGACGAGCAGCAACGCAAACGGTTGGAAGCGTTTCTCACGCAAAAACAGAAAGTAGGTGAGTTGAAGGACGATGATTTCGAGAAGATCTCAGAATTGGGAGCGGGCAACGGTGGTGTCGTTTTCAAAGTTAGTCATAAGCCTAGTGGCCTTGTTATGGCCAGGAAACTCATTCACCTGGAGATTAAGCCTGCCATAAGGAATCAAATTATCAGGGAACTCCAGGTACTGCATGAATGCAACAGTCCCTACATCGTAGGGTTCTACGGAGCGTTTTATTCAGATGGCGAGATCTCCATATGCATGGAGCACATGGATGGGGGGAGTCTCGACCAAGTGCTGAAAAAGGCTGGTAGAATCCCCGAACAAATTTTGGGGAAAGTCTCTATCGCGGTGATCAAAGGTCTGACTTATCTCCGCGAGAAGCACAAAATCATGCATCGGGATGTCAAACCCAGCAACATATTGGTCAATTCAAGAGGTGAGATCAAACTCTGCGATTTTGGAGTTAGTGGGCAACTCATCGACTCTATGGCTAATAGCTTCGTCGGCACTCGAAGTTACATGTCCCCAGAAAGACTGCAAGGTACGCATTACAGTGTTCAAAGCGACATTTGGTCTATGGGCCTTAGCCTCGTTGAGATGGCAGTAGGCAGGTATCCTATTCCGCCGCCGGACGCAAAGGAACTGGAGCTTATGTTCGGTTGCCAGGTGGAGGGGGACGCTGCCGAAACACCTCCCCGACCCAGGACCCCGGGACGCCCACTGAGTTCCTACGGAATGGATAGCCGACCACCCATGGCCATCTTTGAGTTGTTGGATTACATAGTCAACGAACCACCCCCAAAGCTCCCCTCCGGGGTGTTTTCACTCGAATTTCAGGACTTCGTCAACAAGTGCTTGATCAAAAATCCGGCTGAGCGCGCTGATCTTAAACAACTTATGGTTCATGCATTTATCAAACGGAGCGACGCTGAAGAAGTTGACTTTGCAGGCTGGCTCTGTAGCACGATTGGATTGAACCAACCATCCACGCCAACCCACGCAGCGGGGGTTTAG'.upper())

# MEK2
mek2 = Seq('atgctggcccggaggaagccggtgctgccggcgctcaccatcaaccctaccatcgccgagggcccatcccctaccagcgagggcgcctccgaggcaaacctggtggacctgcagaagaagctggaggagctggaacttgacgagcagcagaagaagcggctggaagcctttctcacccagaaagccaaggtcggcgaactcaaagacgatgacttcgaaaggatctcagagctgggcgcgggcaacggcggggtggtcaccaaagtccagcacagaccctcgggcctcatcatggccaggaagctgatccaccttgagatcaagccggccatccggaaccagatcatccgcgagctgcaggtcctgcacgaatgcaactcgccgtacatcgtgggcttctacggggccttctacagtgacggggagatcagcatttgcatggaacacatggatggcggctccctggaccaggtgctgaaagaggccaagaggattcccgaggagatcctggggaaagtcagcatcgcggttctccggggcttggcgtacctccgagagaagcaccagatcatgcaccgagatgtgaagccctccaacatcctcgtgaactctagaggggagatcaagctgtgtgacttcggggtgagcggccagctcatagactccatggccaactccttcgtgggcacgcgctcctacatggctccggagcggttgcagggcacacattactcggtgcagtcggacatctggagcatgggcctgtccctggtggagctggccgtcggaaggtaccccatccccccgcccgacgccaaagagctggaggccatctttggccggcccgtggtcgacggggaagaaggagagcctcacagcatctcgcctcggccgaggccccccgggcgccccgtcagcggtcacgggatggatagccggcctgccatggccatctttgaactcctggactatattgtgaacgagccacctcctaagctgcccaacggtgtgttcacccccgacttccaggagtttgtcaataaatgcctcatcaagaacccagcggagcgggcggacctgaagatgctcacaaaccacaccttcatcaagcggtccgaggtggaagaagtggattttgccggctggttgtgtaaaaccctgcggctgaaccagcccggcacacccacgcgcaccgccgtgTAG'.upper())



In [7]:
# Define genes and check primers against them
genes = {'BRAFnterm':braf_part1,
         'BRAFcterm':braf_part2,
           'KRAS':kras,
           'MRAS':mras,
           'EGFRctermpart1':egfr_cterm_part1,
           'EGFRctermpart2':egfr_cterm_part2,
           'ERBB2ctermpart1':erbb2_cterm_part1,
           'ERBB2ctermpart2':erbb2_cterm_part2,
           'SHP2nterm':shp2_part1,
           'SHP2cterm':shp2_part2,
           'SOS2cterm':sos2_cterm,
           'ARAFnterm':araf_part1,
           'ARAFcterm':araf_part2,
           'CRAFnterm':craf_part1,
           'CRAFcterm':craf_part2,
           'KSR1nterm':ksr1_part1,
           'KSR1cterm':ksr1_part2,
           'KSR2nterm':ksr2_part1,
           'KSR2cterm':ksr2_part2,
           'MEK1':mek1,
           'MEK2':mek2}

forward_primer_array = np.zeros((len(orthogonal_F['PrimerEnd']),len(genes)))
for j, genename in enumerate(genes.keys()):
    print('\n Processing forward primers against ' + genename)
    gene = genes[genename]
    for i, primer in enumerate(orthogonal_F['PrimerEnd']):
        forward_primer_array[i,j] = check_nonspecific(primer, gene)
        
reverse_primer_array = np.zeros((len(orthogonal_R['PrimerEnd']),len(genes)))
for j, genename in enumerate(genes.keys()):
    print('\n Processing reverse primers against ' + genename)
    gene = genes[genename]
    for i, primer in enumerate(orthogonal_R['PrimerEnd']):
        reverse_primer_array[i,j] = check_nonspecific(primer, gene)
        
# Compute number of nonspecific binding sites for each primer
orthogonal_F['Num_Nonspecific_Binding_Sites'] = np.sum(forward_primer_array, axis=1)
orthogonal_R['Num_Nonspecific_Binding_Sites'] = np.sum(reverse_primer_array, axis=1)




 Processing forward primers against BRAFnterm
Found non-specific match using Primer3 at 316bp:
 match:AGGTGCAGAAGGAGAGGAGC
primer:GACCATGCAAGGAGAGGTAC Tm:23.0


/opt/miniconda3/envs/GG_lib/lib/python3.13/site-packages/primer3/bindings.py:305: UserWarning: Function deprecated please use "calc_heterodimer" instead
  return THERMO_ANALYSIS.calcHeterodimer(


Found non-specific match at 1150bp:
 match:AATAGCAGGATCGGCAGCCG
primer:ATAGATCATGTCGGCAGTCG Tm:26.398676349134917

 Processing forward primers against BRAFcterm
Found non-specific match at 104bp:
 match:GTCAGCGGATCGGATCAGGC
primer:TCCAATTATACGGAGCAGGC Tm:22.3
Found non-specific match at 614bp:
 match:CCCTGCGCTGTTTGCCGCGC
primer:AGCTATAAGAATTGCCGGGC Tm:21.587620734826544
Found non-specific match at 856bp:
 match:GACCGCAACGTCGCCGTGCC
primer:ACATTAAATTTCGCCGTGGC Tm:34.10799983199587

 Processing forward primers against KRAS

 Processing forward primers against MRAS

 Processing forward primers against EGFRctermpart1
Found non-specific match at 494bp:
 match:TGCACCGCGACCTGGCAGCC
primer:AATCAGTTTCTTTGGCAGCC Tm:22.0
Found non-specific match at 1053bp:
 match:GCAGGGCTTCTTCAGCAGCC
primer:AATCAGTTTCTTTGGCAGCC Tm:23.1
Found non-specific match at 594bp:
 match:CGCGGTGCACCAAGCGACGG
primer:CTAATATCCCTGAGCGACGG Tm:26.31920458996035
Found non-specific match at 862bp:
 match:TAATTCCTTGATAGCGACGG
prime

In [8]:
# Process primers
orthogonal_F_touse = orthogonal_F[(orthogonal_F.BsaI_Site_Present == False) & \
                                  (orthogonal_F.Num_Nonspecific_Binding_Sites == 0)][['Well Position','PrimerEnd']]
orthogonal_R_touse = orthogonal_R[(orthogonal_R.BsaI_Site_Present == False) & \
                                  (orthogonal_R.Num_Nonspecific_Binding_Sites == 0)][['Well Position','PrimerEnd']]

# Remove primer sequences manually - they bind to one of the genes on the gene list and don't pass oligo qc
primer_seqs_to_remove = ['AGTTGTAATATCACCCGCGC'] #seems to bind to BsaI or SapI sequences, shows up in qc
orthogonal_F_touse_filt = orthogonal_F_touse[~orthogonal_F_touse['PrimerEnd'].isin(primer_seqs_to_remove)]
orthogonal_R_touse_filt = orthogonal_R_touse[~orthogonal_R_touse['PrimerEnd'].isin(primer_seqs_to_remove)]


# concatenate F and R primers into pandas dataframe
orthogonal_primers_touse = pd.concat([orthogonal_F_touse_filt.reset_index().\
                                          drop(['index'], axis=1).rename(columns={'Well Position':'Forward Name',
                                                                         'PrimerEnd':'Forward Primer'}),
                                      orthogonal_R_touse_filt.reset_index().\
                                          drop(['index'], axis=1).rename(columns={'Well Position':'Reverse Name',
                                                                         'PrimerEnd':'Reverse Primer'})],
                                    axis=1)

# remove any primer without a partner
orthogonal_primers_touse = orthogonal_primers_touse.dropna(axis=0)
orthogonal_primers_touse


,Forward Name,Forward Primer,Reverse Name,Reverse Primer
0,A1,AGTATCTCAGCAAGGGCAAC,A1,GTTGCATCTAAGCCAAGTGC
1,B1,CCAGAGCTTAGGGGACATAC,B1,AAGGACTGCATACCAGGTTG
2,C1,GCACGCAAAAGGACATAACC,D1,ACGTGAAACTGTATCGAGCC
3,D1,AGACACAAGGCTGATTCCAG,E1,ATTCAAGGGTTGGACGACTC
4,E1,TCCAATTATACGGAGCAGGC,F1,TACTGATAATTCGGACGCCC
...,...,...,...,...
71,E10,TCGACCAGGTTATCATGAGC,F10,AGGGCTAATTACCATCAGCG
72,G10,ACGATGGGGACATAGAACAC,G10,AGTATTAGGCGTCAAGGTCC
73,H10,GGGCACCGATTAAGAAATGC,H10,AGTTATAAGGGTCCGATGCC
74,A11,ACAAGGAGTCGGCATATCAC,A11,AAGAATTACTGACCCCTCGG


In [9]:
#Import BsaI data
bsaI_empirical = pd.read_csv('./bsaI_empirical.csv')
bsaI_empirical.index = bsaI_empirical['Overhang']
bsaI_empirical = bsaI_empirical.drop(columns=['Overhang'])
bsaI_empirical = bsaI_empirical + 1
bsaI_empirical


,AAAA,AAAC,AAAG,AAAT,AACA,AACC,AACG,AACT,AAGA,AAGC,...,TTCG,TTCT,TTGA,TTGC,TTGG,TTGT,TTTA,TTTC,TTTG,TTTT
Overhang,,,,,,,,,,,,,,,,,,,,,
TTTT,636,9,41,17,3,1,1,1,8,1,...,1,1,1,1,1,1,1,1,1,1
GTTT,4,477,5,46,1,21,1,2,1,16,...,1,1,1,1,1,1,1,1,1,1
CTTT,2,2,597,3,1,1,19,1,1,1,...,1,1,1,1,1,1,1,1,1,1
ATTT,9,5,2,643,1,1,1,7,1,2,...,1,1,1,1,1,1,1,1,1,1
TGTT,1,1,1,1,494,17,65,57,3,1,...,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACAA,1,1,1,1,1,1,1,1,1,1,...,1,1,11,3,8,480,1,1,1,1
TAAA,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,362,1,11,4
GAAA,1,1,1,1,1,1,1,1,1,1,...,1,1,1,3,1,1,6,716,2,20


In [10]:
#Import information about codon usage for mutagenesis
codons_ranked_by_usage = {
    "A": ["GCC", "GCT", "GCA", "GCG"],
    "C": ["TGC", "TGT"],
    "D": ["GAC", "GAT"],
    "E": ["GAG", "GAA"],
    "F": ["TTC", "TTT"],
    "G": ["GGC", "GGA", "GGG", "GGT"],
    "H": ["CAC", "CAT"],
    "I": ["ATC", "ATT", "ATA"],
    "K": ["AAG", "AAA"],
    "L": ["CTG", "CTC", "CTT", "TTG", "TTA", "CTA"],
    "M": ["ATG"],
    "N": ["AAC", "AAT"],
    "P": ["CCC", "CCT", "CCA", "CCG"],
    "Q": ["CAG", "CAA"],
    "R": ["CGG", "AGA", "AGG", "CGC", "CGA", "CGT"],
    "S": ["AGC", "TCC", "TCT", "AGT", "TCA", "TCG"],
    "T": ["ACC", "ACA", "ACT", "ACG"],
    "V": ["GTG", "GTC", "GTT", "GTA"],
    "W": ["TGG"],
    "Y": ["TAC", "TAT"],
}


In [11]:
#Set blacklist of inefficient or nonspecific codons
overhang_blacklist = []
for codon in bsaI_empirical.index.values:
    rc = str(Seq(codon).reverse_complement())   # str, not Seq
    if bsaI_empirical.loc[codon, rc] < 300:
        overhang_blacklist.append(codon)
    elif codon == rc:                            # palindromic overhang self-ligates
        overhang_blacklist.append(codon)
overhang_blacklist


['AATT',
 'ACGT',
 'AGCT',
 'ATAT',
 'GTTG',
 'GGTG',
 'CATG',
 'GTGG',
 'GGGG',
 'GCGG',
 'CCGG',
 'GGCG',
 'CGCG',
 'CTAG',
 'GATC',
 'GCGC',
 'CCGC',
 'GGCC',
 'CGCC',
 'CCCC',
 'CACC',
 'GTAC',
 'CCAC',
 'CAAC',
 'TATA',
 'TCGA',
 'TGCA',
 'TTAA']

In [12]:
# Set defaults
block_size_range = [155, 175]
max_oligo_size = 250
slack = 5
randomsequencepad = "ACGCCGCCACGTGTTCGTTAACTGTTGATTGGTGGCACATAAGTAATACCATGGTCCCTGAAATTCGGCTCAGTTACTTCGAGCGTAATGTCTCAAATGGCGTAGAACGGCAATGACTGTTTGACACTAGGTGGTGTTCAGTTCGGTAACGGAGAGTCTGTGCGGCATTCTTATTAATACATTTGAAACGCGCCCAACTGACGCTAGGCAAGTCAGTGCAGGCTCCCGTGTTAGGATAAGGGTAAACATACAAGTCGATAGAAGATGGGTAGGGGCCTTCAATTCATCCAGCACTCTACG"


In [13]:
def post_qc(amp_primer_set, wt_oligos, primer_set, melt_temp_threshold = 35, check_all_primers=True):
    print("Running QC for primer specificity on WT oligos")
    f_primer_map = {}
    r_primer_map = {}
    # invert the primer to subpool map
    for k, v in amp_primer_set.items():
        f_primer_map[v[1]] = f_primer_map.get(v[1], []) + [k]
        r_primer_map[v[3]] = r_primer_map.get(v[3], []) + [k]
    
    # initialize list of nonspecific problems
    nonspecific = {}
    
    # add unused primers if check_all_primers
    if check_all_primers:
        all_f_primers = np.unique(primer_set['Forward Primer'])
        all_r_primers = np.unique(primer_set['Reverse Primer'])
        for f_primer in all_f_primers:
            if f_primer not in f_primer_map.keys():
                f_primer_map[f_primer] = []
        for r_primer in all_r_primers:
            if r_primer not in r_primer_map.keys():
                r_primer_map[r_primer] = []
        
    for f_primer, subpools_used in f_primer_map.items():
    # iterate over every barcode primer pair and match to each oligo to check for nonspecific amplification
        anneal_locs = []
        for subpoolcheck, fragmentcheck in wt_oligos.items():  # iterate over every WT oligo
            if (subpoolcheck not in subpools_used):  # ignore designed annealing (same name)
                if check_nonspecific(f_primer, fragmentcheck, Tm_rem = melt_temp_threshold, verbose=False) > 0: #use high Tm_rem
                    anneal_locs.append(subpoolcheck)
        if anneal_locs:
            nonspecific.update({f_primer:[a[0] + '_block' + str(a[1]+1) for a in anneal_locs]})
    for r_primer, subpools_used in r_primer_map.items():
    # iterate over every barcode primer pair and match to each oligo to check for nonspecific amplification
        anneal_locs = []
        for subpoolcheck, fragmentcheck in wt_oligos.items():  # iterate over every WT oligo
            if (subpoolcheck not in subpools_used):  # ignore designed annealing (same name)
                if check_nonspecific(r_primer, fragmentcheck, Tm_rem = melt_temp_threshold, verbose=False) > 0: #use high Tm_rem
                    anneal_locs.append(subpoolcheck)
        if anneal_locs:
            nonspecific.update({r_primer:[a[0] + '_block' + str(a[1]+1) for a in anneal_locs]})
    if nonspecific:
        print("Nonspecific Primers: (Manually removing primer sequence recommended)")
        print(nonspecific)
    else:
        print("No non-specific primers detected")
        
    return nonspecific

def build_kmers(sequence, 
                ksize):
    kmers = []
    n_kmers = len(sequence) - ksize + 1

    for i in range(n_kmers):
        kmer = sequence[i:i + ksize]
        kmers.append(kmer)

    return kmers

def compute_overlaps(breakpoints, 
                     inclusion_array, 
                     gene):
    
    overlaps = [[gene[val:val+4].reverse_complement(), gene[val:val+4]] for val in breakpoints]
    counter = 0
    for val in inclusion_array:
        if val == -1:
            (overlaps[counter][1],overlaps[counter+1][0]) = (overlaps[counter+1][0],overlaps[counter][1])
            counter += 1
        elif val == 0:
            overlaps[counter][1] = overlaps[counter+1][1]
            del overlaps[counter+1]
        
    return overlaps

def score_breakpoints(gene, 
                      breakpoint_pair, 
                      empirical, 
                      overhang_blacklist=overhang_blacklist):
    
    #subset empirical matrix by the set of all overlaps
    all_overlaps = []
    for breakpoint in breakpoint_pair:
        all_overlaps.append(gene[breakpoint:(breakpoint+4)])
        all_overlaps.append(gene[breakpoint:(breakpoint+4)].reverse_complement())
    all_overlaps = [str(o) for o in all_overlaps]
    if (len(np.unique(all_overlaps)) == len(all_overlaps)) & (len(set(all_overlaps).intersection(set(overhang_blacklist))) == 0):
        empirical_subset = empirical.loc[all_overlaps,all_overlaps]

        #compute fidelity score
        empirical_subset = empirical_subset/empirical_subset.sum(axis=1)
        fidelity_score = 1
        for breakpoint in breakpoint_pair:
            oh = str(gene[breakpoint:(breakpoint+4)])
            rc = str(gene[breakpoint:(breakpoint+4)].reverse_complement())
            fidelity_score = fidelity_score * empirical_subset.loc[oh, rc]
    
    else:
        fidelity_score = 0
    
    return fidelity_score
    
def optimize_breakpoints(gene, 
                         breakpoint_pair, 
                         indices_to_shift, 
                         indices_of_array,
                         slack, 
                         empirical=bsaI_empirical, 
                         overhang_blacklist=overhang_blacklist):
    
    #compute all enrichments
    shifts = list(range(-slack,slack+1))
    if (len(indices_to_shift) > 2) | (len(indices_to_shift) < 1):
        print('Error -- too many or too few breakpoints!')
        optimum_breakpoint = breakpoint_pair
        optimum_score = 0
    elif (len(indices_to_shift) == 1): #external pair
        scores = [0]*len(shifts)
        for i,shift in enumerate(shifts):
            scores[i] = score_breakpoints(gene, breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shift] + breakpoint_pair[(indices_to_shift[0]+1):], 
                                          empirical=bsaI_empirical, overhang_blacklist=overhang_blacklist)
        
        optimum_shift = np.argmax(scores)
        optimum_breakpoint = breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shifts[optimum_shift]] + breakpoint_pair[(indices_to_shift[0]+1):]
        optimum_score = scores[optimum_shift]
        optimum_length = optimum_breakpoint[indices_of_array[1]] - optimum_breakpoint[indices_of_array[0]]
            
            
    else: #internal pair
        indices_to_shift = sorted(indices_to_shift)
        scores = np.zeros((len(shifts),len(shifts)))
        for i,shift1 in enumerate(shifts):
            for j,shift2 in enumerate(shifts):
                scores[i,j] = score_breakpoints(gene, breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shift1] + \
                                                            breakpoint_pair[(indices_to_shift[0]+1):indices_to_shift[1]] + \
                                                            [breakpoint_pair[indices_to_shift[1]]+shift2] + breakpoint_pair[(indices_to_shift[1]+1):], 
                                              empirical=bsaI_empirical, overhang_blacklist=overhang_blacklist)
                
        optimum_shift = np.unravel_index(np.argmax(scores,axis=None), scores.shape)
        optimum_breakpoint = breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shifts[optimum_shift[0]]] + \
                                            breakpoint_pair[(indices_to_shift[0]+1):indices_to_shift[1]] + \
                                            [breakpoint_pair[indices_to_shift[1]]+shifts[optimum_shift[1]]] + breakpoint_pair[(indices_to_shift[1]+1):]
        optimum_score = scores[optimum_shift]
        optimum_length = optimum_breakpoint[indices_of_array[1]] - optimum_breakpoint[indices_of_array[0]] + 4
    
    return optimum_breakpoint, optimum_score, optimum_length

def optimize_gene(gene, 
                  block_size_range=block_size_range, 
                  slack=slack, 
                  empirical=bsaI_empirical, 
                  overhang_blacklist=overhang_blacklist): 
    
    #setup initial inputs to optimization
    gene_size = len(gene)
    protein_size = len(gene.translate())
        
    #exclude gene if it is too big
    if protein_size > 1000:
        print('Protein size too big!')
        
    #divide genes between 500 and 1000aa into two blocks
    elif protein_size > 620:
        print('Protein size too big! Will add two superblock (620aa+ proteins) soon.')
        
    else:
        #gene is one superblock 
        #print('Protein is one superblock.')
        block_size = block_size_range[0] + np.argmin(
            [abs(gene_size/(i+block_size_range[0])-round(gene_size/(i+block_size_range[0]))) \
                 for i in range(0, block_size_range[1]-block_size_range[0])])
        fragment_number = int(np.ceil(gene_size/block_size))
        first_breakpoint = 0
        last_breakpoint = gene_size-4
        initial_breakpoints = [[first_breakpoint,block_size+slack+2,last_breakpoint]] + \
                                [[0,i-2-slack,i+block_size+slack+2,last_breakpoint] for i in range(block_size, gene_size-block_size-slack-4, block_size)] + \
                                [[0,gene_size-2-slack-block_size,last_breakpoint]]
    
        #optimize each breakpoint
        optimum_breakpoints = []
        optimum_scores = []
        optimum_lengths = []
        oligo_array_indices = []
        for k,breakpoint in enumerate(initial_breakpoints):
            if len(breakpoint) == 3:
                indices_of_array = [0, 1] if k==0 else [1, 2]
                optimum_breakpoint, optimum_score, optimum_length = optimize_breakpoints(gene, breakpoint, [1], indices_of_array,
                                                            slack, empirical=bsaI_empirical, overhang_blacklist=overhang_blacklist)
                optimum_breakpoints.append(optimum_breakpoint)
                optimum_scores.append(optimum_score)
                optimum_lengths.append(optimum_length)
                oligo_array_indices.append(indices_of_array)
            else:
                indices_of_array = [1, 2]
                optimum_breakpoint, optimum_score, optimum_length = optimize_breakpoints(gene, breakpoint, [1, 2], indices_of_array,
                                                            slack, empirical=bsaI_empirical, overhang_blacklist=overhang_blacklist)
                optimum_breakpoints.append(optimum_breakpoint)
                optimum_scores.append(optimum_score)
                optimum_lengths.append(optimum_length)
                oligo_array_indices.append(indices_of_array)
    
    optimum_overlaps = [[str(gene[t:(t+4)]) for t in s] for s in optimum_breakpoints]
    if all([s >= 0.95 for s in optimum_scores]):
        print('All regions are high fidelity!')
    elif all([s >= 0.9 for s in optimum_scores]):
        print('Some regions are medium fidelity.')
    else:
        print('Some regions are low fidelity. Look closer')
        
    return optimum_breakpoints, optimum_overlaps, optimum_scores, optimum_lengths, oligo_array_indices

def generate_primer(DNA_seq,
                     Fwd=True,
                     extendtoCG=False,
                     smallest_primer_size=16,
                     largest_primer_size=30,
                     Tm=55):
    
    #Setup melting temperature arrays
    melt_temp_array = np.zeros(largest_primer_size-smallest_primer_size+1)
    
    if Fwd:
        DNA_seq_touse = DNA_seq
    else:
        DNA_seq_touse = DNA_seq.reverse_complement()
            
    #Make melting temperature arrays
    primer_length = 0
    for i in range(smallest_primer_size,largest_primer_size+1):
        melt_temp_array[i-smallest_primer_size] = mt.Tm_NN(DNA_seq_touse[0:i])
        
        #Pick F primer when Tm is first >F_Tm
        if (melt_temp_array[i-smallest_primer_size] >= Tm) & (primer_length==0):
            primer_length = i
    
    #If Tm isnt high enough after max bases, just set primer length to be max and hope it works
    if (primer_length == 0):
        primer_length = largest_primer_size
        
    if extendtoCG:
        while ((DNA_seq_touse[primer_length-1] == 'A') | (DNA_seq_touse[primer_length-1] == 'T')) & \
                    (primer_length < largest_primer_size):
            primer_length += 1
    
    return DNA_seq_touse[0:primer_length]

def make_all_mutations(region_name,
                       region,
                       region_flanks=[Seq(''),Seq('')],
                       nt_start=0, #zero-indexed!
                       wt_only=False,
                       synonymous=True,
                       stops='TAA',
                       all3ntdeletions=True,
                       codons_ranked_by_usage=codons_ranked_by_usage):
    
    oligo_array = {}
    #Check that region has size divisible by three
    if (len(region)/3 != len(region)//3) | (nt_start/3 != nt_start//3):
        print('Region is not translatable!')
        
    else:
        #add wt seq to oligo array
        oligo_name = region_name + '_WT'
        wt_seq = \
            region_flanks[0] + region + region_flanks[1]
        oligo_array[oligo_name] = wt_seq
        
        if not wt_only:
                    
            #loop over amino acids
            for j in range(0,len(region),3):

                #add all missense variants
                aa = region[j:(j+3)].translate()
                for aa_to in codons_ranked_by_usage.keys():
                    if aa_to != aa:
                        oligo_name = region_name + '_' + str(aa) + str((nt_start+j)//3+1) + str(aa_to)
                        seq_to_append = \
                            region_flanks[0] + \
                            region[0:j] + Seq(codons_ranked_by_usage[aa_to][0]) + \
                            region[(j+3):] + \
                            region_flanks[1]
                        oligo_array[oligo_name] = seq_to_append

                #add synonymous variant if True and if possible, 
                # using the most common codon that is NOT the codon in the gene
                if synonymous:
                    if len(codons_ranked_by_usage[aa]) > 1:
                        oligo_name = region_name + '_' + str(aa) + str((nt_start+j)//3+1) + str(aa)
                        possible_codons = codons_ranked_by_usage[aa].copy()
                        possible_codons.remove(region[j:(j+3)])
                        seq_to_append = \
                            region_flanks[0] + \
                            region[0:j] + Seq(possible_codons[0]) + \
                            region[(j+3):] + \
                            region_flanks[1]
                        oligo_array[oligo_name] = seq_to_append

                #add stops if true
                if stops:
                    oligo_name = region_name + '_' + str(aa) + str((nt_start+j)//3+1) + 'X'
                    seq_to_append = \
                        region_flanks[0] + \
                        region[0:j] + Seq(stops) + \
                        region[(j+3):] + \
                        region_flanks[1]
                    oligo_array[oligo_name] = seq_to_append

                #add all 3nt deletions if True
                if all3ntdeletions:
                    for k in range(0,3):
                        if j+k+3 <= len(region):
                            oligo_name = region_name + '_' + 'del' + str(nt_start+j+k+1)
                            seq_to_append = \
                                region_flanks[0] + \
                                region[0:(j+k)] + \
                                region[(j+k+3):] + \
                                region_flanks[1]
                            oligo_array[oligo_name] = seq_to_append
        
    return oligo_array


def write_oligo_library(genes,
                        oligo_file='./oligo_test.csv',
                        primer_file='./primer_test.tsv',
                        gbl_file='./gbl_test.tsv',
                        amp_primer_key_file='./amp_primer_key.tsv',
                        primer_set=orthogonal_primers_touse,
                        codons_ranked_by_usage=codons_ranked_by_usage,
                        block_size_range=block_size_range, 
                        max_oligo_size=max_oligo_size,
                        slack=slack, 
                        empirical=bsaI_empirical, 
                        overhang_blacklist=overhang_blacklist,
                        wt_only=False,
                        synonymous=True,
                        stops='TAA',
                        all3ntdeletions=True,
                        smallest_primer_size=16,
                        largest_primer_size=30,
                        Tm=55,
                        extendtoCG=True,
                        bsaI_firstoverlap='CGTC',
                        bsaI_lastoverlap='GCAT',
                        all_blocks=True,
                        blocks_to_include=False,
                        sapIcapF=True,
                        sapIcapR=True,
                        check_all_primers=True,
                        qc_melt_temp_threshold=32,
                        randomsequencepad=randomsequencepad):
    
    #Split up primer set into F and R primers, cannot do more than 82 sublibraries
    oligo_primer_counter = 0
    oligo_array = {}
    amp_primers = {}
    gblocks = {}
    num_blocks = {}
    amp_primer_dict = {}
    
    #Convert genes to Seq and genes to list
    gene_names = list(genes.keys())
    genes = [Seq(genes[gene_name].upper()) for gene_name in gene_names]
    
    #SapI and BsaI site sequences
    sapI_seq = Seq('GCTCTTC')
    sapI_seqplusone = Seq('GCTCTTCC')
    bsaI_seq = Seq('GGTCTC')
    bsaI_seqplusone = Seq('GGTCTCT')
    pcr_capseq = Seq('GGCTAC') + bsaI_seqplusone
    gbl_capseq_F = Seq('CCGCGTGATTACGAGTCG') + pcr_capseq
    gbl_capseq_R = Seq('GGGTTAGCAAGTGGCAGCCT') + pcr_capseq
    
    # iterate over genes
    for r,gene in enumerate(genes):
        
        print('Processing gene ' + str(r+1))
        gene_name = gene_names[r]
        
        #exclude if gene size is not divisible by three
        if len(gene)/3 != len(gene)//3:
            print('Gene length is not divisible by 3!')
    
        #exclude if there is a SapI site in the gene
        elif any([True for kmer in build_kmers(gene, len(sapI_seq)) if kmer==sapI_seq]) | \
            any([True for kmer in build_kmers(gene.reverse_complement(), len(sapI_seq)) if kmer==sapI_seq]):
            print('Gene has SapI site!')
        
        #exclude if there is a BsaI site in the gene
        elif any([True for kmer in build_kmers(gene, len(bsaI_seq)) if kmer==bsaI_seq]) | \
            any([True for kmer in build_kmers(gene.reverse_complement(), len(bsaI_seq)) if kmer==bsaI_seq]):
            print('Gene has BsaI site!')
            
        else:
            print('Gene has no SapI or BsaI sites! Performing GoldenGate optimization...')
            
            #cap gene with BsaI breakpoints and possible SapI sites 
            if sapIcapF & sapIcapR:
                gene_capped = bsaI_firstoverlap + sapI_seqplusone + \
                        gene + sapI_seqplusone.reverse_complement() + bsaI_lastoverlap
                capping_length_F = len(bsaI_firstoverlap + sapI_seqplusone)
                capping_length_R = len(sapI_seqplusone.reverse_complement() + bsaI_lastoverlap)
            elif sapIcapF:
                gene_capped = bsaI_firstoverlap + sapI_seqplusone + \
                        gene + bsaI_lastoverlap
                capping_length_F = len(bsaI_firstoverlap + sapI_seqplusone)
                capping_length_R = len(bsaI_lastoverlap)
            elif sapIcapR:
                gene_capped = bsaI_firstoverlap + \
                        gene + sapI_seqplusone.reverse_complement() + bsaI_lastoverlap
                capping_length_F = len(bsaI_firstoverlap)
                capping_length_R = len(sapI_seqplusone.reverse_complement() + bsaI_lastoverlap)
            else:
                gene_capped = bsaI_firstoverlap + gene + bsaI_lastoverlap
                capping_length_F = len(bsaI_firstoverlap)
                capping_length_R = len(bsaI_lastoverlap)
            
            #Optimize gene
            optimum_breakpoints, optimum_overlaps, optimum_scores, optimum_lengths, oligo_array_indices = \
            optimize_gene(gene_capped, 
                      block_size_range=block_size_range, 
                      slack=slack, 
                      empirical=bsaI_empirical, 
                      overhang_blacklist=overhang_blacklist)
            pprint.pprint({'Optimum Breakpoints': optimum_breakpoints, 
                   'Optimum Overlaps': optimum_overlaps, 
                   'Optimum Scores': optimum_scores})
            num_blocks[gene_name] = len(optimum_breakpoints)
            
            #add primers for gene_F and gene_R that are repeated constantly throughout the PCRs
            #note: should probably prevalidate these primers!
            F_primer = generate_primer(gene,
                                       Fwd=True,
                                       extendtoCG=extendtoCG,
                                       smallest_primer_size=smallest_primer_size,
                                       largest_primer_size=largest_primer_size,
                                       Tm=Tm)
            F_primer = pcr_capseq + bsaI_firstoverlap + sapI_seqplusone + F_primer
            amp_primers[gene_name+'_gene'+'_ampF'] = F_primer
            R_primer = generate_primer(gene,
                                       Fwd=False,
                                       extendtoCG=extendtoCG,
                                       smallest_primer_size=smallest_primer_size,
                                       largest_primer_size=largest_primer_size,
                                       Tm=Tm)
            R_primer = pcr_capseq + Seq(bsaI_lastoverlap).reverse_complement() + \
                        sapI_seqplusone + R_primer
            amp_primers[gene_name+'_gene'+'_ampR'] = R_primer
            
            #make oligos, primers, gblocks for each block
            for i,breakpoint in enumerate(optimum_breakpoints):
                
                #find indices of breakpoint that correspond to oligo vs need to be PCRed/gblock
                pcr_indices = [[j,j+1] for j in range(len(breakpoint)-1)]
                pcr_indices.remove(oligo_array_indices[i])
                
                #find mutagenic window of oligo
                oligo_breaks = [breakpoint[j] for j in oligo_array_indices[i]]
                oligo_mutagenic_window = [int(3*np.ceil(max(oligo_breaks[0]+4-capping_length_F,3)/3)), int(3*np.floor(min(oligo_breaks[1]-capping_length_F,len(gene)-1)/3))]
                
                #subset the right block if needed
                if (all_blocks == True) | ((i+1) in blocks_to_include[r] if blocks_to_include != False else True): #subset on allowed blocks
                
                    #add pcr primers and gblocks, one segment at a time
                    for k,pcr_index in enumerate(pcr_indices):
                        piece_name = gene_name + '_block' + str(i+1) + '_s' + str(k+1)
                        pcr_breaks = [breakpoint[j] for j in pcr_index]
                                            
                        #get pcr primers
                        if pcr_breaks[0] == breakpoint[0]: #Fragment beginning at gene start 
                            R_primer = generate_primer(gene_capped[pcr_breaks[0]:(pcr_breaks[1]+4)],
                                                       Fwd=False,
                                                       extendtoCG=extendtoCG,
                                                       smallest_primer_size=smallest_primer_size,
                                                       largest_primer_size=largest_primer_size,
                                                       Tm=Tm)
                            R_primer = pcr_capseq + R_primer
                            amp_primers[piece_name+'_ampR'] = R_primer
                        elif pcr_breaks[1] == breakpoint[-1]: #Fragment ending at gene end
                            F_primer = generate_primer(gene_capped[pcr_breaks[0]:(pcr_breaks[1]+4)],
                                                       Fwd=True,
                                                       extendtoCG=extendtoCG,
                                                       smallest_primer_size=smallest_primer_size,
                                                       largest_primer_size=largest_primer_size,
                                                       Tm=Tm)
                            F_primer = pcr_capseq + F_primer
                            amp_primers[piece_name+'_ampF'] = F_primer
                        else:
                            F_primer = generate_primer(gene_capped[pcr_breaks[0]:(pcr_breaks[1]+4)],
                                                       Fwd=True,
                                                       extendtoCG=extendtoCG,
                                                       smallest_primer_size=smallest_primer_size,
                                                       largest_primer_size=largest_primer_size,
                                                       Tm=Tm)
                            F_primer = pcr_capseq + F_primer
                            R_primer = generate_primer(gene_capped[pcr_breaks[0]:(pcr_breaks[1]+4)],
                                                       Fwd=False,
                                                       extendtoCG=extendtoCG,
                                                       smallest_primer_size=smallest_primer_size,
                                                       largest_primer_size=largest_primer_size,
                                                       Tm=Tm)
                            R_primer = pcr_capseq + R_primer
                            amp_primers[piece_name+'_ampF'] = F_primer
                            amp_primers[piece_name+'_ampR'] = R_primer

                        #make gblocks
                        gbl = gbl_capseq_F + gene_capped[pcr_breaks[0]:(pcr_breaks[1]+4)] + gbl_capseq_R.reverse_complement()
                        gblocks[piece_name] = gbl
                        
                    #add oligos to oligo array
                    name_primer_F, primer_F, name_primer_R, primer_R = \
                            primer_set.iloc[oligo_primer_counter,][['Forward Name',
                                                                  'Forward Primer',
                                                                  'Reverse Name',
                                                                  'Reverse Primer']]
                    add_on_array = make_all_mutations(gene_name + '_block' + str(i+1),
                                       gene[oligo_mutagenic_window[0]:oligo_mutagenic_window[1]],
                                       region_flanks=[Seq(primer_F) + \
                                                      bsaI_seqplusone + \
                                                      gene_capped[oligo_breaks[0]:(oligo_mutagenic_window[0]+capping_length_F)] ,
                                                      gene_capped[(oligo_mutagenic_window[1]+capping_length_F):(oligo_breaks[1]+4)] + \
                                                      bsaI_seqplusone.reverse_complement() + \
                                                      Seq(primer_R).reverse_complement()],
                                       nt_start=oligo_mutagenic_window[0],
                                       wt_only=wt_only,
                                       synonymous=synonymous,
                                       stops=stops,
                                       all3ntdeletions=all3ntdeletions,
                                       codons_ranked_by_usage=codons_ranked_by_usage)
                    oligo_array.update(add_on_array)
                    amp_primer_dict.update({(gene_name,i+1): (name_primer_F,primer_F,name_primer_R,primer_R)})
                    oligo_primer_counter += 1
                    
    #Check that max oligo is less than the max oligo length
    if sum([len(s)>max_oligo_size for s in oligo_array.values()]) == 0:
        print('All oligos are below the maximum 250bp!')
    else:
        print('Some oligos are TOO BIG!')
        
    #Check for nonspecific amplification
    wt_oligos = {tuple([key.split('_')[0],
                        int((key.split('_block')[1]).split('_')[0])]
                      ):oligo_array[key] \
                     for key in oligo_array.keys() if 'WT' in key}
    nonspecific_primers = post_qc(amp_primer_dict, 
                                  wt_oligos,
                                  primer_set, 
                                  melt_temp_threshold=qc_melt_temp_threshold,
                                  check_all_primers=check_all_primers)
                
    #Remove any oligos with additional BsaI sites or SapI sites
    bad_oligos = []
    for name,oligo in oligo_array.items():
        #check for sapI sites
        sapI_F = sum([True for kmer in build_kmers(oligo, len(sapI_seq)) if kmer==sapI_seq])
        sapI_R = sum([True for kmer in build_kmers(oligo.reverse_complement(), len(sapI_seq)) if kmer==sapI_seq])
        #check that oligo is block 1 if it contains a sapI site in the forward orientation 
        if sapI_F > 0:
            if ('block1' not in name) | (sapI_F > 1):
                bad_oligos.append(name)
        #check that the oligo is block final if it contains a sapI site in the reverse orientation
        if sapI_R > 0:
            if ('block'+str(num_blocks[name.split('_')[0]]) not in name) | (sapI_R > 1):
                bad_oligos.append(name)
        #check for more than one BsaI site
        bsaI_F = sum([True for kmer in build_kmers(oligo, len(bsaI_seq)) if kmer==bsaI_seq])
        bsaI_R = sum([True for kmer in build_kmers(oligo.reverse_complement(), len(bsaI_seq)) if kmer==bsaI_seq])
        if (bsaI_F != 1) | (bsaI_R != 1):
            bad_oligos.append(name)
    for oligo_name in bad_oligos:
        del oligo_array[oligo_name]
    print(str(len(bad_oligos)) + ' oligos deleted due to errant restriction sites.')
    
    #Remove any duplicate oligos
    new_dict = {}
    seen_values = set()
    counter=0
    for key, value in oligo_array.items():
        if value not in seen_values:
            new_dict[key] = value
            seen_values.add(value)
        else:
            counter += 1
    print(str(counter) + ' oligos removed due to duplication.')
    oligo_array = new_dict
    del new_dict
    
    #write oligo array to file
    with open(oligo_file, 'w') as f:
        for key in oligo_array.keys():
            f.write("%s,%s\n"%(key,oligo_array[key]))
    f.close()
            
    #write primers to file
    primer_order_sheet = []
    for key in amp_primers.keys():
        primer_order_sheet.append(key + '\t' + \
                 str(amp_primers[key]) + \
                 '\t' + '25nm' + '\t' + 'STD\n')
    print(*primer_order_sheet)
    with open(primer_file, 'w') as f:
        for line in primer_order_sheet:
            f.write(line)
    f.close()
    
    #write amplification primer key to file
    amp_primer_key = ['Gene' + '\t' + 'Block' + '\t' + \
                      'Forward Primer Well' + '\t' + 'Forward Primer' + '\t' + \
                      'Reverse Primer Well' + '\t' + 'Reverse Primer' + '\n']
    for key in amp_primer_dict.keys():
        genename, geneblock = key[0], str(key[1])
        name_primer_F, primer_F, name_primer_R, primer_R = amp_primer_dict[key]
        amp_primer_key.append(genename + '\t' + geneblock + '\t' + \
                 name_primer_F + '\t' + primer_F + '\t' + \
                 name_primer_R + '\t' + primer_R + '\n')
    print(*amp_primer_key)
    with open(amp_primer_key_file, 'w') as f:
        for line in amp_primer_key:
            f.write(line)
    f.close()
    
    #write gblocks to file
    gblock_order_sheet = []
    for key in gblocks.keys():
        # pad gblock if it is not 300bp for Twist
        if len(gblocks[key]) < 300:
            gblocks[key] = Seq(randomsequencepad[0:(300-len(gblocks[key]))]) + gblocks[key]
        gblock_order_sheet.append(key + '\t' + \
                 str(gblocks[key]) + '\n')
    print(*gblock_order_sheet)
    with open(gbl_file, 'w') as f:
        for line in gblock_order_sheet:
            f.write(line)
    f.close()
    
    return oligo_array,amp_primers,gblocks,amp_primer_dict
                

In [14]:
# make new combinations by cyclically permuting only the reverse primers by 25 and 50
reverse_shifted1 = pd.concat([orthogonal_primers_touse[['Reverse Name','Reverse Primer']].iloc[18:].reset_index(),
                               orthogonal_primers_touse[['Reverse Name','Reverse Primer']].iloc[:18].reset_index()],
                              axis=0).drop('index', axis=1).reset_index()
new_combos_shifted1 = pd.concat([orthogonal_primers_touse[['Forward Name','Forward Primer']],
                                 reverse_shifted1],axis=1).drop('index', axis=1)
                                 
reverse_shifted2 = pd.concat([orthogonal_primers_touse[['Reverse Name','Reverse Primer']].iloc[36:].reset_index(),
                               orthogonal_primers_touse[['Reverse Name','Reverse Primer']].iloc[:36].reset_index()],
                              axis=0).drop('index', axis=1).reset_index()
new_combos_shifted2 = pd.concat([orthogonal_primers_touse[['Forward Name','Forward Primer']],
                                 reverse_shifted2],axis=1).drop('index', axis=1)

reverse_shifted3 = pd.concat([orthogonal_primers_touse[['Reverse Name','Reverse Primer']].iloc[54:].reset_index(),
                               orthogonal_primers_touse[['Reverse Name','Reverse Primer']].iloc[:54].reset_index()],
                              axis=0).drop('index', axis=1).reset_index()
new_combos_shifted3 = pd.concat([orthogonal_primers_touse[['Forward Name','Forward Primer']],
                                 reverse_shifted3],axis=1).drop('index', axis=1)

all_combos = pd.concat([orthogonal_primers_touse,
                       new_combos_shifted1,
                       new_combos_shifted2,
                       new_combos_shifted3], axis=0).reset_index().drop('index', axis=1)
all_combos


,Forward Name,Forward Primer,Reverse Name,Reverse Primer
0,A1,AGTATCTCAGCAAGGGCAAC,A1,GTTGCATCTAAGCCAAGTGC
1,B1,CCAGAGCTTAGGGGACATAC,B1,AAGGACTGCATACCAGGTTG
2,C1,GCACGCAAAAGGACATAACC,D1,ACGTGAAACTGTATCGAGCC
3,D1,AGACACAAGGCTGATTCCAG,E1,ATTCAAGGGTTGGACGACTC
4,E1,TCCAATTATACGGAGCAGGC,F1,TACTGATAATTCGGACGCCC
...,...,...,...,...
299,E10,TCGACCAGGTTATCATGAGC,E7,TCATCGACAAGATACAGGCG
300,G10,ACGATGGGGACATAGAACAC,F7,ATTGACGGGAACTACACTCG
301,H10,GGGCACCGATTAAGAAATGC,G7,TGAGCCATAAAAGCAAAGCG
302,A11,ACAAGGAGTCGGCATATCAC,H7,AGACAACAATCTGAGGCTGG


In [15]:
# Create BRAF (1-400aa), KRAS, CRAF (1-336) libraries - testv2
oligo_array,amp_primers,gblocks,amp_primer_dict = write_oligo_library({'BRAFnterm':braf_part1,
                                                                       'KRAS':kras,
                                                                       'CRAFnterm':craf_part1},
                                                                      oligo_file='./pilot_order_011324/circRNApilot_oligos_v3.csv',
                                                                      primer_file='./pilot_order_011324/circRNApilot_primers_v3.tsv',
                                                                      gbl_file='./pilot_order_011324/circRNApilot_gblocks_v3.tsv',
                                                                      amp_primer_key_file='./pilot_order_011324/circRNApilot_ampkey_v3.tsv',
                                                                      block_size_range=[152,173],
                                                                      primer_set=all_combos)


Processing gene 1
Gene has no SapI or BsaI sites! Performing GoldenGate optimization...
Some regions are medium fidelity.
{'Optimum Breakpoints': [[0, np.int64(166), 1298],
                         [0, 153, np.int64(336), 1298],
                         [0, 324, np.int64(491), 1298],
                         [0, 481, np.int64(661), 1298],
                         [0, 646, np.int64(818), 1298],
                         [0, 813, np.int64(984), 1298],
                         [0, 966, np.int64(1150), 1298],
                         [0, np.int64(1137), 1298]],
 'Optimum Overlaps': [['CGTC', 'AGAT', 'GCAT'],
                      ['CGTC', 'TGGA', 'TTCT', 'GCAT'],
                      ['CGTC', 'AATG', 'CCTG', 'GCAT'],
                      ['CGTC', 'TTCG', 'TTAC', 'GCAT'],
                      ['CGTC', 'CGGA', 'CCAG', 'GCAT'],
                      ['CGTC', 'TTCC', 'AGTA', 'GCAT'],
                      ['CGTC', 'TCTG', 'ACGA', 'GCAT'],
                      ['CGTC', 'CCTG', 'GCAT']],
 'Op

/opt/miniconda3/envs/GG_lib/lib/python3.13/site-packages/primer3/bindings.py:305: UserWarning: Function deprecated please use "calc_heterodimer" instead
  return THERMO_ANALYSIS.calcHeterodimer(


No non-specific primers detected
112 oligos deleted due to errant restriction sites.
834 oligos removed due to duplication.
BRAFnterm_gene_ampF	GGCTACGGTCTCTCGTCGCTCTTCCATGGCTGCGCTGAGCG	25nm	STD
 BRAFnterm_gene_ampR	GGCTACGGTCTCTATGCGCTCTTCCCCTTTCTCGTTGAGGTCCAGGG	25nm	STD
 BRAFnterm_block1_s1_ampF	GGCTACGGTCTCTAGATGATTAAACTGACGCAAGAGCATATTG	25nm	STD
 BRAFnterm_block2_s1_ampR	GGCTACGGTCTCTTCCACACCTCCTCAGGAATAGC	25nm	STD
 BRAFnterm_block2_s2_ampF	GGCTACGGTCTCTTTCTCAGTTAGTTCCAGTGCCAGTATG	25nm	STD
 BRAFnterm_block3_s1_ampR	GGCTACGGTCTCTCATTGCCCAGGCTTTCAAGGAG	25nm	STD
 BRAFnterm_block3_s2_ampF	GGCTACGGTCTCTCCTGCCAAACAAACAAAGAACAGTTG	25nm	STD
 BRAFnterm_block4_s1_ampR	GGCTACGGTCTCTCGAACAATCGGCTTTTGTGGGC	25nm	STD
 BRAFnterm_block4_s2_ampF	GGCTACGGTCTCTTTACAGGGGAGGAGCTTCACG	25nm	STD
 BRAFnterm_block5_s1_ampR	GGCTACGGTCTCTTCCGTGTCCCAACCTATTGGTTTTTTC	25nm	STD
 BRAFnterm_block5_s2_ampF	GGCTACGGTCTCTCCAGCGGTGCTCCACAGAG	25nm	STD
 BRAFnterm_block6_s1_ampR	GGCTACGGTCTCTGGAATTTATACCCGCAGGTCTGAC	25nm	S